In [1]:
import pandas as pd
import vivarium_inputs
import vivarium.gbd_mapping as gbd_mapping
import pathlib
from lsff_utils import config_utils
from lsff_utils.results import expand_to_all_scenarios, aggregate_by_cause_and_scenario

In [2]:
location = "india"
vehicle = "rice"

In [3]:
# Parameters
location = "india"
vehicle = "rice"


In [4]:
scenarios = list(
    config_utils.get_location_fortificant_vehicle_intervention_scenarios()
    .pipe(lambda df: df[(df.location == location) & (df.vehicle == vehicle)])
    .intervention_scenario.unique()
) + ["zero", "baseline"]
scenarios

['intervention', 'zero', 'baseline']

In [5]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/ylls.parquet"
if pathlib.Path(path).is_file():
    pregnancy_ylls = pd.read_parquet(path)
else:
    pregnancy_ylls = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/ylls.parquet"
        ).assign(value=0),
        scenarios,
    )
pregnancy_ylls

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,ylls,cause,other_causes,other_causes,10_to_14,invalid,1,baseline,0,2,0.0
1,ylls,cause,other_causes,other_causes,10_to_14,invalid,2,baseline,0,2,0.0
2,ylls,cause,other_causes,other_causes,10_to_14,invalid,3,baseline,0,2,0.0
3,ylls,cause,other_causes,other_causes,10_to_14,invalid,4,baseline,0,2,0.0
4,ylls,cause,other_causes,other_causes,10_to_14,invalid,5,baseline,0,2,0.0
...,...,...,...,...,...,...,...,...,...,...,...
26995,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,1,baseline,0,9,0.0
26996,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,2,baseline,0,9,0.0
26997,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,3,baseline,0,9,0.0
26998,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,4,baseline,0,9,0.0


In [6]:
pregnancy_ylls.groupby("scenario").random_seed.nunique()

scenario
baseline        10
intervention    10
zero            10
Name: random_seed, dtype: int64

In [7]:
assert (pregnancy_ylls[pregnancy_ylls.value > 0].entity == "maternal_disorders").all()

In [8]:
pregnancy_ylls_by_scenario = aggregate_by_cause_and_scenario(pregnancy_ylls).pipe(
    lambda df: df[df.index.get_level_values("entity") == "maternal_disorders"]
)
pregnancy_ylls_by_scenario

scenario      entity              wealth_quintile
baseline      maternal_disorders  1                  413159.272588
                                  2                  245276.294149
                                  3                  312863.941400
                                  4                  249072.466042
                                  5                  174991.786434
intervention  maternal_disorders  1                  400459.321830
                                  2                  242620.113177
                                  3                  309528.646489
                                  4                  249072.466042
                                  5                  169049.616707
zero          maternal_disorders  1                  422333.247189
                                  2                  250562.470712
                                  3                  319137.428004
                                  4                  255097.746825
            

In [9]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/ylds.parquet"
if pathlib.Path(path).is_file():
    pregnancy_ylds = pd.read_parquet(path)
else:
    pregnancy_ylds = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/ylds.parquet"
        ).assign(value=0),
        scenarios,
    )

pregnancy_ylds

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,ylds,cause,pregnancy,pregnant,10_to_14,invalid,1,baseline,0,2,0.0
1,ylds,cause,pregnancy,parturition,10_to_14,invalid,1,baseline,0,2,0.0
2,ylds,cause,pregnancy,postpartum,10_to_14,invalid,1,baseline,0,2,0.0
3,ylds,cause,maternal_disorders,maternal_disorders,10_to_14,invalid,1,baseline,0,2,0.0
4,ylds,cause,maternal_hemorrhage,maternal_hemorrhage,10_to_14,invalid,1,baseline,0,2,0.0
...,...,...,...,...,...,...,...,...,...,...,...
94495,ylds,cause,pregnancy,postpartum,95_plus,severe,5,baseline,0,9,0.0
94496,ylds,cause,maternal_disorders,maternal_disorders,95_plus,severe,5,baseline,0,9,0.0
94497,ylds,cause,maternal_hemorrhage,maternal_hemorrhage,95_plus,severe,5,baseline,0,9,0.0
94498,ylds,cause,all_causes,all_causes,95_plus,severe,5,baseline,0,9,0.0


In [10]:
# Pregnancy has no disability, and maternal hemorrhage disability is counted in maternal_disorders
assert (
    pregnancy_ylds[
        pregnancy_ylds.entity.isin(["pregnancy", "maternal_hemorrhage"])
    ].value
    == 0
).all()

In [11]:
pregnancy_ylds_by_scenario = aggregate_by_cause_and_scenario(pregnancy_ylds).pipe(
    lambda df: df[
        ~df.index.get_level_values("entity").isin(["pregnancy", "maternal_hemorrhage"])
    ]
)
pregnancy_ylds_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              1                   96933.934137
                                  2                   76368.635983
                                  3                   60526.979136
                                  4                   51885.973679
                                  5                   40635.113550
              maternal_disorders  1                   73788.627278
                                  2                   39037.683527
                                  3                   45313.580946
                                  4                   45429.735462
                                  5                   31340.265551
intervention  anemia              1                   93781.069258
                                  2                   73558.608397
                                  3                   57889.705739
                                  4                   49389.367337
            

In [12]:
pregnancy_dalys_by_scenario = pregnancy_ylls_by_scenario.add(
    pregnancy_ylds_by_scenario, fill_value=0
)
pregnancy_dalys_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              1                   96933.934137
                                  2                   76368.635983
                                  3                   60526.979136
                                  4                   51885.973679
                                  5                   40635.113550
              maternal_disorders  1                  486947.899866
                                  2                  284313.977676
                                  3                  358177.522347
                                  4                  294502.201505
                                  5                  206332.051985
intervention  anemia              1                   93781.069258
                                  2                   73558.608397
                                  3                   57889.705739
                                  4                   49389.367337
            

In [13]:
ylds_path = f"results/rescaled_child_results/{vehicle}/{location}/ylds.parquet"
if pathlib.Path(ylds_path).is_file():
    assert (pd.read_parquet(ylds_path)['value'] == 0).all()

In [14]:
path = f"results/rescaled_child_results/{vehicle}/{location}/ylls.parquet"

# NOTE: The child_scenario column currently contains only 'baseline'
# because we didn't have any interventions in the child simulation. If
# we add a child intervention that creates another scenario in this
# column, then results from different child scenarios would get added
# together in the call to aggregate_by_cause_and_scenario below, so we'd
# need to change the processing code in that case.
def assert_unique_child_scenario(df):
    assert set(df.child_scenario.unique()) == {'baseline'}
    return df

if pathlib.Path(path).is_file():
    neonatal_ylls = (
        pd.read_parquet(path)
        .pipe(assert_unique_child_scenario)
        .rename(columns={"maternal_scenario": "scenario"})
    )
else:
    # NOTE: This else branch is for processing the Ethiopia results,
    # where no Vivarium sims were run, so the corresponding DALYs should
    # just be 0
    neonatal_ylls = expand_to_all_scenarios(
        pd.read_parquet(f"results/rescaled_child_results/rice/india/ylls.parquet")
        .assign(value=0)
        .rename(columns={"maternal_scenario": "scenario"}),
        scenarios,
    )

neonatal_ylls

,measure,entity_type,entity,sub_entity,age_group,sex,wealth_quintile,child_scenario,scenario,input_draw,random_seed,value
0,ylls,cause,other_causes,other_causes,0_to_5_months,Female,1,baseline,intervention,0,2,697077.816041
1,ylls,cause,other_causes,other_causes,0_to_5_months,Female,2,baseline,intervention,0,2,519567.456628
2,ylls,cause,other_causes,other_causes,0_to_5_months,Female,3,baseline,intervention,0,2,575765.137541
3,ylls,cause,other_causes,other_causes,0_to_5_months,Female,4,baseline,intervention,0,2,463225.127240
4,ylls,cause,other_causes,other_causes,0_to_5_months,Female,5,baseline,intervention,0,2,454591.265371
...,...,...,...,...,...,...,...,...,...,...,...,...
1195,ylls,cause,other_causes,other_causes,18_to_59_months,Male,1,baseline,intervention,0,6,75276.583283
1196,ylls,cause,other_causes,other_causes,18_to_59_months,Male,2,baseline,intervention,0,6,62943.046988
1197,ylls,cause,other_causes,other_causes,18_to_59_months,Male,3,baseline,intervention,0,6,54762.885237
1198,ylls,cause,other_causes,other_causes,18_to_59_months,Male,4,baseline,intervention,0,6,37750.489006


In [15]:
neonatal_ylls_by_scenario = aggregate_by_cause_and_scenario(neonatal_ylls)
assert (
    neonatal_ylls_by_scenario[
        neonatal_ylls_by_scenario.index.get_level_values("entity") != "other_causes"
    ]
    == 0
).all()
neonatal_ylls_by_scenario = neonatal_ylls_by_scenario[
    neonatal_ylls_by_scenario.index.get_level_values("entity") == "other_causes"
]
neonatal_ylls_by_scenario = (
    neonatal_ylls_by_scenario.reset_index()
    .assign(entity="lbwsg")
    .set_index(neonatal_ylls_by_scenario.index.names)
    .value
)
neonatal_ylls_by_scenario

scenario      entity  wealth_quintile
baseline      lbwsg   1                  1.784370e+07
                      2                  1.471134e+07
                      3                  1.345889e+07
                      4                  1.232743e+07
                      5                  1.209394e+07
intervention  lbwsg   1                  1.783071e+07
                      2                  1.468535e+07
                      3                  1.343722e+07
                      4                  1.229711e+07
                      5                  1.208094e+07
zero          lbwsg   1                  1.786536e+07
                      2                  1.474599e+07
                      3                  1.347188e+07
                      4                  1.234909e+07
                      5                  1.209393e+07
Name: value, dtype: float64

In [16]:
path = f"../0400_non_pregnant_anemia_model/results/{vehicle}/{location}/ylds.parquet"
if pathlib.Path(path).is_file():
    non_pregnancy_anemia_ylds = pd.read_parquet(path)
else:
    non_pregnancy_anemia_ylds = expand_to_all_scenarios(
        pd.read_parquet(
            f"../0400_non_pregnant_anemia_model/results/rice/india/ylds.parquet"
        ).assign(value=0),
        scenarios,
    )

non_pregnancy_anemia_ylds

,sex,age_start,age_end,wealth_quintile,value,scenario
0,Female,0.0,0.019178,1,1186.490411,zero
1,Female,0.0,0.019178,2,901.389895,zero
2,Female,0.0,0.019178,3,798.971870,zero
3,Female,0.0,0.019178,4,664.417092,zero
4,Female,0.0,0.019178,5,465.048259,zero
...,...,...,...,...,...,...
745,Male,95.0,125.000000,1,76.740774,intervention
746,Male,95.0,125.000000,2,71.124606,intervention
747,Male,95.0,125.000000,3,71.983957,intervention
748,Male,95.0,125.000000,4,66.963850,intervention


In [17]:
# For comparison with previous round of results, we also look at
# WRA and U5
wra_non_pregnancy_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds[
        (non_pregnancy_anemia_ylds.sex == "Female")
        & (non_pregnancy_anemia_ylds.age_start >= 10)
        & (non_pregnancy_anemia_ylds.age_end <= 55)
    ].assign(entity="anemia", input_draw="draw_0")
)
wra_non_pregnancy_anemia_ylds_by_scenario

scenario      entity  wealth_quintile
baseline      anemia  1                  1.820498e+06
                      2                  1.783737e+06
                      3                  1.823285e+06
                      4                  1.736274e+06
                      5                  1.581907e+06
intervention  anemia  1                  1.745983e+06
                      2                  1.704613e+06
                      3                  1.732228e+06
                      4                  1.644322e+06
                      5                  1.476570e+06
zero          anemia  1                  1.935722e+06
                      2                  1.894235e+06
                      3                  1.926864e+06
                      4                  1.828468e+06
                      5                  1.629746e+06
Name: value, dtype: float64

In [18]:
scenarios[1]

'zero'

In [19]:
(
    wra_non_pregnancy_anemia_ylds_by_scenario.loc["baseline"].sum()
    + pregnancy_ylds_by_scenario.loc[("baseline", "anemia")].sum()
) - (
    wra_non_pregnancy_anemia_ylds_by_scenario.loc[scenarios[1]].sum()
    + pregnancy_ylds_by_scenario.loc[(scenarios[1], "anemia")].sum()
)

-498684.6767363604

In [20]:
u5_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds[(non_pregnancy_anemia_ylds.age_end <= 5)].assign(
        entity="anemia", input_draw="draw_0"
    )
)
u5_anemia_ylds_by_scenario

scenario      entity  wealth_quintile
baseline      anemia  1                  625607.175667
                      2                  511243.694931
                      3                  471320.145264
                      4                  402165.769700
                      5                  314579.489511
intervention  anemia  1                  600288.851980
                      2                  490021.004135
                      3                  450461.421272
                      4                  383096.198481
                      5                  296712.837232
zero          anemia  1                  657232.983445
                      2                  537238.145707
                      3                  492948.098560
                      4                  420532.322873
                      5                  323175.185458
Name: value, dtype: float64

In [21]:
(
    u5_anemia_ylds_by_scenario.loc["baseline"].sum()
    - u5_anemia_ylds_by_scenario.loc[scenarios[1]].sum()
)

-106210.46097104298

In [22]:
non_pregnancy_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds.assign(entity="anemia", input_draw="draw_0")
)
non_pregnancy_anemia_ylds_by_scenario

scenario      entity  wealth_quintile
baseline      anemia  1                  4.385539e+06
                      2                  3.984875e+06
                      3                  3.930992e+06
                      4                  3.617826e+06
                      5                  3.225927e+06
intervention  anemia  1                  4.204414e+06
                      2                  3.807838e+06
                      3                  3.736778e+06
                      4                  3.427995e+06
                      5                  3.014606e+06
zero          anemia  1                  4.664061e+06
                      2                  4.232154e+06
                      3                  4.153325e+06
                      4                  3.809427e+06
                      5                  3.321418e+06
Name: value, dtype: float64

In [23]:
path = f"../0500_neural_tube_defects_model/results/{location}/{vehicle}/ylls_by_scenario.csv"
if pathlib.Path(path).is_file():
    neural_tube_defect_ylls_by_scenario = pd.read_csv(path)
else:
    neural_tube_defect_ylls_by_scenario = expand_to_all_scenarios(
        pd.read_csv(
            f"../0500_neural_tube_defects_model/results/india/rice/intervention/ylls_by_scenario.csv"
        ).assign(value=0),
        scenarios,
    )

neural_tube_defect_ylls_by_scenario = neural_tube_defect_ylls_by_scenario.set_index(
    ["scenario", "entity", "wealth_quintile"]
).value
neural_tube_defect_ylls_by_scenario

scenario      entity  wealth_quintile
zero          ntd     1                  362852.984954
                      2                  320542.743777
                      3                  287772.048247
                      4                  271354.859542
                      5                  231101.543094
baseline      ntd     1                  346981.666189
                      2                  309124.934471
                      3                  278190.908238
                      4                  262984.391222
                      5                  228110.398677
intervention  ntd     1                  211851.488633
                      2                  199425.758699
                      3                  177714.962874
                      4                  168831.743401
                      5                  163986.328606
Name: value, dtype: float64

In [24]:
dalys_by_scenario = (
    pregnancy_dalys_by_scenario.add(neonatal_ylls_by_scenario, fill_value=0)
    .add(non_pregnancy_anemia_ylds_by_scenario, fill_value=0)
    .add(neural_tube_defect_ylls_by_scenario, fill_value=0)
)
dalys_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              1                  4.482473e+06
                                  2                  4.061244e+06
                                  3                  3.991519e+06
                                  4                  3.669712e+06
                                  5                  3.266562e+06
              lbwsg               1                  1.784370e+07
                                  2                  1.471134e+07
                                  3                  1.345889e+07
                                  4                  1.232743e+07
                                  5                  1.209394e+07
              maternal_disorders  1                  4.869479e+05
                                  2                  2.843140e+05
                                  3                  3.581775e+05
                                  4                  2.945022e+05
                          

In [25]:
import pathlib

In [26]:
path = f"./results/{location}/{vehicle}/dalys_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
dalys_by_scenario.to_csv(path)